# Advanced Measurements and Calibration Workflows

Use the root `ChipCalibration.ipynb` for daily flat-configuration edits.
This notebook uses parameterized experiments that can also be exposed to Blueprint. Every `run()` performs a hardware acquisition.
Session settings come from `lab/project.yaml` and the calibration store. Session does not automatically read `config_all` from another notebook.
For custom pulse development, use `Measurement.run(Program, run_cfg)` in the main notebook.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "lab/project.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from QickworkspaceV2 import Session

from QickworkspaceV2 import save_labber_results


In [ ]:
session = Session.from_project(ROOT / "lab/project.yaml")

## MUX / PYNQ Readout-Frequency Sweep

Select the profile matching the actual wiring in `project.yaml`. The host loop updates the static tone registers at each frequency point while preserving tone-slot assignments.

In [ ]:
result = session.run("resonator_spec", target="Q1,Q2", start=-5, stop=5, points=101)
labber_paths = save_labber_results(result, load=session.load)
result.plot()

## Readout Spectroscopy in the e State

In [ ]:
result = session.run("resonator_spec_e", target="Q1", start=-5, stop=5, points=101)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## Readout Spectroscopy in the f State

In [ ]:
result = session.run("resonator_spec_f", target="Q1", start=-5, stop=5, points=101)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## GE repeated Rabi

In [ ]:
result = session.scan("power_rabi_chevron_ge", "iterations", [1, 3, 5, 7], target="Q1")
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## EF repeated Rabi

In [ ]:
result = session.scan("power_rabi_chevron_ef", "iterations", [1, 3, 5, 7], target="Q1")
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## GE DRAG

In [ ]:
result = session.scan("drag_ge", "alpha", [-0.4, -0.2, 0.0, 0.2, 0.4], target="Q1", delta_mhz=-200)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## EF DRAG

In [ ]:
result = session.scan("drag_ef", "alpha", [-0.4, -0.2, 0.0, 0.2, 0.4], target="Q1", delta_mhz=-200)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## AllXY

In [ ]:
result = session.run("allxy", target="Q1,Q2")
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## Randomized benchmarking

In [ ]:
result = session.run("randomized_benchmarking", target="Q1")
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## Conditional Ramsey

In [ ]:
result = session.run("conditional_ramsey", target="Q1,Q2")
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## Coupler chevron

In [ ]:
result = session.run("coupler_chevron", target="Q1,Q2", gain_points=21, length_points=41)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## Joint State Tomography

Requires measured readout thresholds committed to the calibration store.

In [ ]:
result = session.run("state_tomography", target="Q1,Q2", reps=2000)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## QICK bias resonator flux

In [ ]:
result = session.scan("resonator_flux", "bias_gain", [0.0, 0.05, 0.1], target="Q1", bias_port="coupler_c12", bias_length_us=100)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## TWPA transmission

In [ ]:
result = session.run("twpa_probe", target="Q1", points=101)
labber_paths = save_labber_results(result, load=session.load)
print(result.metrics)
result.plot()

## External Instruments

The procedures in `lab/procedures/resonator_dc_flux.py`, `twpa_pump_power.py`, `twpa_pump_frequency.py`, and `twpa_flux.py` are maintained independently.
`InstrumentAxis` binds actual instrument read/write methods, physical units, and limits. The scan records readback values and attempts to restore the initial setting on completion or interruption; restoration failures are recorded and reported.
Example:

```python
from lab.procedures.twpa_pump_power import run
# Turn the pump off before acquiring a reference on the same frequency grid.
reference = session.run("twpa_probe", target="Q1", points=101)
# Configure and enable the actual pump, then bind the driver's read/write methods.
measured, gain = run(
    session, [-30, -25, -20], read_power=pump.get_power, set_power=pump.set_power,
    minimum_dbm=-40, maximum_dbm=0, resource_id="pump", reference=reference,
    target="Q1", points=101,
)
```

## Calibration and NVIDIA Blueprint

`session.propose(result)` creates a reviewable set of parameter updates. `session.commit(proposal)` writes those updates to the calibration store.
Choose updates appropriate to the measured experiment. A two-dimensional chevron plot does not establish a calibrated CZ gate.

```python
proposal = session.propose(result)
print(proposal.updates)
# After reviewing the updates, run session.commit(proposal) in a separate cell.
```

Start the worker with `qickworkspace serve`. UI, CLI and Agent read the live worker `GET /catalog`; no experiment wrappers are required.
The connection, data directory, and hardware profile come from `lab/project.yaml`. See `docs/WORKER.md` for the complete integration contract.